# Transfomer 

逐步实现 Transformer 完整架构：

1. 实现主体网络, 可输入和输出
2. 实现输入层
3. 实现 encoder
4. 实现 decoder
5. 实现 输出层

## Config 

In [2]:
# for debug 

dim = 512 
num_layers = 6
heads = 8

batch_size = 2
src_len = 256
trg_len = 128
max_len = 512

src_vocab_size = 100
trg_vocab_size = 200

# N = 2048 # dataset

IGNORE_INDEX = -100

## Transformer 

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(42)

In [16]:
class TransformerBasic(nn.Module):
    """
    仅通过 Embedding 和 Lienar 实现 Transformer 计算逻辑
    输入:[bs, src_seq_len]
    输出:[bs, trg_seq_len, vocab_size]
    """
    def __init__(self, src_vocab_size = 100, trg_vocab_size = 200, dim = 512, num_layers = 6, heads = 8, max_len = 512):
        super().__init__()
        self.encoder_input = nn.Embedding(src_vocab_size, dim)
        self.encoder = nn.Linear(dim, dim)
        
        self.decoder_input = nn.Embedding(trg_vocab_size, dim)
        self.decoder = nn.Linear(dim, dim)

        self.output_layer = nn.Linear(dim, trg_vocab_size)
        
    def forward(self, src_ids, trg_ids, src_mask = None, trg_mask = None, src_trg_mask = None):
        X = self.encoder_input(src_ids)
        X_src = self.encoder(X)
        print('encoder output:\t', X_src.shape)

        Y = self.decoder_input(trg_ids)
        Y = self.decoder(Y) + X_src.mean(dim = 1, keepdim = True)
        print('decoder output:\t', Y.shape)

        logits = self.output_layer(Y) 
        prob = F.softmax(logits, dim = -1)
        
        return logits, prob
    
model = TransformerBasic()
print(model)


src_ids = torch.randint(src_vocab_size, (batch_size, src_len))
trg_ids = torch.randint(trg_vocab_size, (batch_size, trg_len))
print('encode input shape: ', src_ids.shape)
print('decode output shape: ', trg_ids.shape)

logits, _ = model(src_ids, trg_ids)
print('transformer output:\t', logits.shape)

TransformerBasic(
  (encoder_input): Embedding(100, 512)
  (encoder): Linear(in_features=512, out_features=512, bias=True)
  (decoder_input): Embedding(200, 512)
  (decoder): Linear(in_features=512, out_features=512, bias=True)
  (output_layer): Linear(in_features=512, out_features=200, bias=True)
)
encode input shape:  torch.Size([2, 256])
decode output shape:  torch.Size([2, 128])
encoder output:	 torch.Size([2, 256, 512])
decoder output:	 torch.Size([2, 128, 512])
transformer output:	 torch.Size([2, 128, 200])


- Decoder 输入 `bs x trg_len` 与 输出 ` bs x trg_len x trg_vocab_size`, 一个序列输出 `trg_len` 个 概率分布

# Transformer Input Layer

In [13]:
torch.arange(0, 10, 2)
a = torch.tensor([1,2,3]) # a is row
b = torch.tensor([2,2,2]) # b is col
torch.outer(a, b)

tensor([[2, 2, 2],
        [4, 4, 4],
        [6, 6, 6]])

In [27]:
class TransformerInputLayer(nn.Module):
    """
    词向量 + 位置编码
    """
    def __init__(self, vocab_size = 100, dim = 512, max_len = 1024, base = 10000.0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, dim)
        self.position_encoding = nn.Parameter( torch.randn(max_len, dim) ) # learnable
        self.max_len = max_len

        # sin-cos position encoding
        # 人工设计的 位置编码
        group = dim // 2
        theta_ids = torch.arange(0, dim, 2) # 0, 2, 4, ..., 512
        theta =  1 / ( base ** ( theta_ids / dim ) )
        pe = torch.zeros(dim) # 512, sin( theta_0 ),cos( theta_0), ...
        pe[theta_ids] = theta
        pe[theta_ids+1] = theta

        position_ids = torch.arange(0, max_len) # 0, 1, 2, ..., 1024
        self.PE = torch.outer(position_ids, pe) # 1024 x 512
        
        self.PE[:, theta_ids] = torch.sin(self.PE[:, theta_ids])
        self.PE[:, theta_ids+1] = torch.sin(self.PE[:, theta_ids+1])

    def forward_NoPE(self, input_ids):
        """
        嵌入向量 + 无位置编码
        """
        X = self.embedding(input_ids)
        return X

    def forward_basic(self, input_ids):
        """
        嵌入向量 + 常数向量位置编码
        """
        bs, seq_len = input_ids.shape
        X = self.embedding(input_ids)
        PE = torch.arange(seq_len).unsqueeze(dim = 0).unsqueeze(dim = 2)
        X_ = X + PE / self.max_len
        return X_

    def forward_learn(self, input_ids):
        """
        嵌入向量 + 可学习位置编码
        """
        bs, seq_len = input_ids.shape
        X = self.embedding(input_ids)
        PE = self.position_encoding[:seq_len, :]
        X_ = X + PE
        return X_

    def forward(self, input_ids):
        """
        嵌入向量 + 绝对位置编码(标准实现)
        """
        bs, seq_len = input_ids.shape
        X = self.embedding(input_ids)
        PE = self.PE[:seq_len, :]
        X_ = X + PE
        return X_

input_layer = TransformerInputLayer(vocab_size = src_vocab_size, dim = 6)
print('NoPE: ', input_layer.forward_NoPE(src_ids[:1, :3]))
print('Constant: ', input_layer.forward_basic(src_ids[:1, :3]))
print('Learnable: ', input_layer.forward_learn(src_ids[:1, :3]))
print('Sin-cos-PE: ', input_layer.forward(src_ids[:1, :3]))

NoPE:  tensor([[[ 1.3571,  0.7318, -0.3634, -1.4031, -0.0480, -2.5067],
         [ 1.2771, -0.7329,  1.2579, -1.1189,  0.7377, -1.0855],
         [ 1.6461, -3.3568,  0.3720, -0.4821,  1.3213, -2.0404]]],
       grad_fn=<EmbeddingBackward0>)
Constant:  tensor([[[ 1.3571,  0.7318, -0.3634, -1.4031, -0.0480, -2.5067],
         [ 1.2781, -0.7320,  1.2589, -1.1179,  0.7387, -1.0845],
         [ 1.6480, -3.3549,  0.3740, -0.4802,  1.3233, -2.0384]]],
       grad_fn=<AddBackward0>)
Learnable:  tensor([[[ 2.3714,  2.2573, -0.8149, -0.4895,  0.7280, -2.7798],
         [ 1.7801, -0.3407, -0.6924, -1.5115,  1.4592, -1.1352],
         [ 1.9481, -3.7913, -0.3975, -0.8799,  0.9189, -1.7315]]],
       grad_fn=<AddBackward0>)
Sin-cos-PE:  tensor([[[ 1.3571,  0.7318, -0.3634, -1.4031, -0.0480, -2.5067],
         [ 2.1186,  0.1085,  1.3043, -1.0725,  0.7398, -1.0834],
         [ 2.5554, -2.4475,  0.4647, -0.3894,  1.3256, -2.0361]]],
       grad_fn=<AddBackward0>)


## Transformer Encoder

- 归一化层
- 多头注意力层
- 前馈层
- 残差链接

### LayerNorm

In [34]:
class LayerNorm(nn.Module):
    def __init__(self, dim, ):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))
        self.epsilon = 1e-8
    def forward(self, X, ):
        mu = X.mean( dim = -1, keepdim = True)
        var = X.var( dim = -1, keepdim = True)
        X_hat = ( X - mu ) / torch.sqrt( var + self.epsilon)
        Y = X_hat * self.gamma + self.beta
        return Y
        
tmp_dim = 4
LN = LayerNorm(dim = tmp_dim)
print(LN)
X = torch.randn(2,3,4)
print(LN(X))

LayerNorm()
tensor([[[-0.3820,  0.7669, -1.2393,  0.8544],
         [ 0.8011, -1.2336, -0.3933,  0.8257],
         [-0.8519, -0.8275,  1.1392,  0.5402]],

        [[ 1.1172, -0.0655,  0.2469, -1.2987],
         [ 0.9063,  0.7887, -0.6024, -1.0926],
         [-0.1115,  0.6192, -1.3664,  0.8587]]], grad_fn=<AddBackward0>)


### Feed Forward Network(FFN)

In [35]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, dim, ):
        super().__init__()
        self.dim = dim
        self.W_up = nn.Linear(self.dim, 4 * self.dim)
        self.ReLU = nn.ReLU()
        self.W_down = nn.Linear(4 * self.dim, self.dim)
    def forward(self, X):
        X_ = self.ReLU(self.W_up(X))
        Y = self.W_down(X_)
        return Y
        
FFN = FeedForwardNetwork(dim = 4)
X = torch.randn(2,3,4)
print(FFN)
print(FFN(X))

FeedForwardNetwork(
  (W_up): Linear(in_features=4, out_features=16, bias=True)
  (ReLU): ReLU()
  (W_down): Linear(in_features=16, out_features=4, bias=True)
)
tensor([[[ 0.4506,  0.1309,  0.6478, -0.4604],
         [ 0.1240,  0.1878,  0.1652, -0.0496],
         [ 0.0990,  0.2915,  0.0935,  0.0927]],

        [[ 0.2589,  0.1926,  0.3157, -0.2385],
         [ 0.3874,  0.1267,  0.2108, -0.2653],
         [ 0.5566,  0.0988,  0.7540, -0.4216]]], grad_fn=<ViewBackward0>)


### Multi Heads Attention

self-attention

In [70]:
import math

class MultiHeadScaleDotProductAttention(nn.Module):
    def __init__(self, dim_in, dim_out, heads = 8):
        super().__init__()
        self.WQ = nn.Linear(dim_in, dim_out)
        self.WK = nn.Linear(dim_in, dim_out)
        self.WV = nn.Linear(dim_in, dim_out)
        self.WO = nn.Linear(dim_in, dim_out)
        self.heads = 8
        self.head_dim = dim_out // self.heads
        
    def forward(self, X_Q, X_K, X_V, mask = None):
        bs, seq_len, dim = X_Q.shape
        bs, seq_K_len, dim = X_K.shape
        bs, seq_V_len, dim = X_V.shape
        Q = self.WQ(X_Q)
        K = self.WK(X_K)
        V = self.WV(X_V)

        # 拆分维度
        Q_h = Q.view(bs, seq_len, self.heads, self.head_dim).transpose(1,2)
        K_h = K.view(bs, seq_K_len, self.heads, self.head_dim).transpose(1,2) # KV len 可以不等同于 Q len
        V_h = V.view(bs, seq_V_len, self.heads, self.head_dim).transpose(1,2)

        # 多个 q_i 计算注意力特征
        S = Q_h @ K_h.transpose(2,3) / math.sqrt(self.head_dim) # 1. 为什么要除于 \sqrt{d}

        if mask is not None:
            idx = torch.where(mask == 0)
            S[:, idx[0],idx[1],idx[2]] = -10000.0
        
        P = torch.softmax(S, dim = -1) # 行 softmax
        Z = P @ V_h

        # 恢复维度
        Z = Z.transpose(1,2).reshape(bs, seq_len, dim)
        
        output = self.WO(Z)
        
        return output
tmp_dim = 16
X = torch.randn(2, 4, tmp_dim)
mask = torch.ones(2, 4, tmp_dim)
model = MultiHeadScaleDotProductAttention(tmp_dim, tmp_dim, 8)
Y = model(X, X, X, mask)
print(X.shape, Y.shape)

torch.Size([2, 4, 16]) torch.Size([2, 4, 16])


### Encoder Block

In [71]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, dim = 512, num_layers = 6, heads = 8):
        super().__init__()
        self.attn = MultiHeadScaleDotProductAttention(dim, dim, heads)
        self.ln1 = LayerNorm(dim)
        self.ffn = FeedForwardNetwork(dim)
        self.ln2 = LayerNorm(dim)
        
    def forward(self, X, src_mask = None):
        X_attn = self.attn(X, X, X)
        X_ln = self.ln1(X_attn)
        X = X + X_ln

        X_ffn = self.ffn(X)
        X_ln = self.ln2(X_ffn)
        X = X + X_ln

        return X

tmp_dim = 16
X = torch.randn(2, 4, tmp_dim)
mask = torch.ones(2, 4, tmp_dim)
model = TransformerEncoderBlock(tmp_dim, tmp_dim, 8)
Y = model(X, mask)
print(X.shape, Y.shape)

torch.Size([2, 4, 16]) torch.Size([2, 4, 16])


In [72]:
class TransformerEncoder(nn.Module):
    """
    输入 原文本序列，输出 token 序列的编码表征
    输入:[bs, src_seq_len, dim]
    输出:[bs, src_seq_len, dim]
    """
    def __init__(self, vocab_size = 100, dim = 512, num_layers = 6, heads = 8):
        super().__init__()
        # self.encoder = nn.Linear(dim, dim) 
        self.encoder = nn.ModuleList(
            [TransformerEncoderBlock(dim, heads) for i in range(num_layers)]
        )
    def forward(self, X, mask = None):
        for encode_block in self.encoder:
            X = encode_block(X, mask)
        return X
        
tmp_dim = 16
X = torch.randn(2, 4, tmp_dim)
mask = torch.ones(2, 4, tmp_dim)
model = TransformerEncoder(tmp_dim, tmp_dim, 8)
Y = model(X, mask)
print(X.shape, Y.shape)

torch.Size([2, 4, 16]) torch.Size([2, 4, 16])


In [73]:
print(model)

TransformerEncoder(
  (encoder): ModuleList(
    (0-7): 8 x TransformerEncoderBlock(
      (attn): MultiHeadScaleDotProductAttention(
        (WQ): Linear(in_features=16, out_features=16, bias=True)
        (WK): Linear(in_features=16, out_features=16, bias=True)
        (WV): Linear(in_features=16, out_features=16, bias=True)
        (WO): Linear(in_features=16, out_features=16, bias=True)
      )
      (ln1): LayerNorm()
      (ffn): FeedForwardNetwork(
        (W_up): Linear(in_features=16, out_features=64, bias=True)
        (ReLU): ReLU()
        (W_down): Linear(in_features=64, out_features=16, bias=True)
      )
      (ln2): LayerNorm()
    )
  )
)


### Encoder Mask detail

In [74]:
def get_src_mask(input_ids, pad_token_id = 0):
    bs, seq_len = input_ids.shape
    mask = torch.ones(bs, seq_len, seq_len)
    for i in range(bs):
        pad_idx =  torch.where(input_ids[i, :]  == 0)[0]
        mask[i, pad_idx, :] = 0
        mask[i, :, pad_idx] = 0
    return mask
    
input_ids = torch.tensor([[1, 2, 3, 0, 0],
                          [1, 2, 0, 0, 0]], dtype = torch.long) # 0 is pad
mask = get_src_mask(input_ids, pad_token_id = 0)
print(mask)

score = torch.randn(bs, seq_len, seq_len)
score * mask

tensor([[[1., 1., 1., 0., 0.],
         [1., 1., 1., 0., 0.],
         [1., 1., 1., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[1., 1., 0., 0., 0.],
         [1., 1., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]]])


tensor([[[ 0.3627,  0.6510,  0.8498, -0.0000, -0.0000],
         [ 0.3896,  1.6035, -0.5014, -0.0000, -0.0000],
         [-0.3026, -0.2000,  0.5631,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
         [-0.0000, -0.0000,  0.0000,  0.0000, -0.0000]],

        [[-0.5440,  0.0864, -0.0000,  0.0000, -0.0000],
         [ 0.3905,  1.3143, -0.0000, -0.0000,  0.0000],
         [-0.0000,  0.0000,  0.0000, -0.0000, -0.0000],
         [-0.0000, -0.0000, -0.0000,  0.0000, -0.0000],
         [-0.0000,  0.0000,  0.0000, -0.0000,  0.0000]]])

In [75]:
multi_head_score = torch.randn(bs, 2, seq_len, seq_len) # multi-head score
multi_head_score * mask.unsqueeze(1)

tensor([[[[ 4.9182e-01, -4.6709e-01,  1.1316e-03,  0.0000e+00, -0.0000e+00],
          [-8.2204e-03, -9.3396e-01,  1.1204e+00,  0.0000e+00, -0.0000e+00],
          [-1.0505e-03, -1.8984e-01, -3.7614e-01,  0.0000e+00, -0.0000e+00],
          [ 0.0000e+00, -0.0000e+00,  0.0000e+00, -0.0000e+00,  0.0000e+00],
          [-0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00]],

         [[-1.6098e+00, -1.6920e+00,  1.9565e+00, -0.0000e+00, -0.0000e+00],
          [ 5.5313e-01,  2.6433e-01,  8.2760e-01, -0.0000e+00,  0.0000e+00],
          [ 1.0043e+00,  6.0289e-01, -2.3585e-01,  0.0000e+00,  0.0000e+00],
          [-0.0000e+00, -0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00],
          [-0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00]]],


        [[[ 2.0020e-01, -1.8283e+00, -0.0000e+00,  0.0000e+00,  0.0000e+00],
          [-1.8867e-01,  2.6850e-01, -0.0000e+00,  0.0000e+00,  0.0000e+00],
          [ 0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000

## Transformer Decoder

In [88]:
class TransformerDecoderBlock(nn.Module):
    def __init__(self, dim = 512, num_layers = 6, heads = 8):
        super().__init__()
        self.masked_attn = MultiHeadScaleDotProductAttention(dim, dim, heads)
        self.ln1 = LayerNorm(dim)
    
        self.cross_attn = MultiHeadScaleDotProductAttention(dim, dim, heads)
        self.ln2 = LayerNorm(dim)
        
        self.ffn = FeedForwardNetwork(dim)
        self.ln3 = LayerNorm(dim)
        
    def forward(self, X, X_src, trg_mask = None, src_trg_mask = None):
        X_attn = self.masked_attn(X, X, X, trg_mask)
        X_ln = self.ln1(X_attn)
        X = X + X_ln
        
        X_attn = self.cross_attn(X, X_src, X_src, src_trg_mask)
        X_ln = self.ln2(X_attn)
        X = X + X_ln

        X_ffn = self.ffn(X)
        X_ln = self.ln3(X_ffn)
        X = X + X_ln

        return X

tmp_dim = 16
X = torch.randn(2, 4, tmp_dim)
X_src = torch.randn(2, 8, tmp_dim)
mask = torch.ones(2, 4, tmp_dim)
model = TransformerDecoderBlock(tmp_dim, tmp_dim, 8)
print(model)
Y = model(X, X_src, X_src)
print(X.shape, Y.shape)

TransformerDecoderBlock(
  (masked_attn): MultiHeadScaleDotProductAttention(
    (WQ): Linear(in_features=16, out_features=16, bias=True)
    (WK): Linear(in_features=16, out_features=16, bias=True)
    (WV): Linear(in_features=16, out_features=16, bias=True)
    (WO): Linear(in_features=16, out_features=16, bias=True)
  )
  (ln1): LayerNorm()
  (cross_attn): MultiHeadScaleDotProductAttention(
    (WQ): Linear(in_features=16, out_features=16, bias=True)
    (WK): Linear(in_features=16, out_features=16, bias=True)
    (WV): Linear(in_features=16, out_features=16, bias=True)
    (WO): Linear(in_features=16, out_features=16, bias=True)
  )
  (ln2): LayerNorm()
  (ffn): FeedForwardNetwork(
    (W_up): Linear(in_features=16, out_features=64, bias=True)
    (ReLU): ReLU()
    (W_down): Linear(in_features=64, out_features=16, bias=True)
  )
  (ln3): LayerNorm()
)
torch.Size([2, 4, 16]) torch.Size([2, 4, 16])


### masked-self-attention mask detail

In [79]:
def get_trg_mask(input_ids, pad_token_id = 0):
    bs, seq_len = input_ids.shape
    mask = torch.tril(torch.ones(bs, seq_len, seq_len)) # tril
    for i in range(bs):
        pad_idx =  torch.where(input_ids[i, :]  == 0)[0]
        mask[i, pad_idx, :] = 0
        mask[i, :, pad_idx] = 0
    return mask
    
input_ids = torch.tensor([[1, 2, 3, 0, 0],
                          [1, 2, 0, 0, 0]], dtype = torch.long) # 0 is pad
mask = get_trg_mask(input_ids, pad_token_id = 0)
print(mask)

score = torch.randn(bs, seq_len, seq_len)
score * mask

tensor([[[1., 0., 0., 0., 0.],
         [1., 1., 0., 0., 0.],
         [1., 1., 1., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[1., 0., 0., 0., 0.],
         [1., 1., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]]])


tensor([[[ 1.4802,  0.0000,  0.0000, -0.0000, -0.0000],
         [-0.5539, -1.8003,  0.0000, -0.0000, -0.0000],
         [ 0.0871,  1.1515, -0.4287, -0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000, -0.0000,  0.0000],
         [ 0.0000, -0.0000,  0.0000,  0.0000,  0.0000]],

        [[-2.5649,  0.0000, -0.0000, -0.0000,  0.0000],
         [-0.3734, -0.2268,  0.0000, -0.0000, -0.0000],
         [-0.0000, -0.0000,  0.0000, -0.0000,  0.0000],
         [ 0.0000, -0.0000,  0.0000, -0.0000,  0.0000],
         [-0.0000,  0.0000,  0.0000,  0.0000, -0.0000]]])

In [80]:
multi_head_score = torch.randn(bs, 2, seq_len, seq_len) # multi-head score
multi_head_score * mask.unsqueeze(1)

tensor([[[[ 0.6476,  0.0000, -0.0000,  0.0000, -0.0000],
          [ 1.0998, -1.2148,  0.0000, -0.0000,  0.0000],
          [ 0.6644, -1.7969, -0.1551,  0.0000,  0.0000],
          [-0.0000, -0.0000, -0.0000, -0.0000,  0.0000],
          [ 0.0000, -0.0000,  0.0000,  0.0000,  0.0000]],

         [[-0.1258, -0.0000, -0.0000,  0.0000, -0.0000],
          [ 0.3050, -0.1712,  0.0000, -0.0000,  0.0000],
          [ 1.4754, -0.1692, -0.2104, -0.0000,  0.0000],
          [-0.0000,  0.0000,  0.0000, -0.0000, -0.0000],
          [ 0.0000, -0.0000, -0.0000,  0.0000,  0.0000]]],


        [[[-0.9780, -0.0000,  0.0000,  0.0000, -0.0000],
          [-0.3437, -1.0601, -0.0000, -0.0000, -0.0000],
          [-0.0000,  0.0000,  0.0000,  0.0000, -0.0000],
          [ 0.0000,  0.0000,  0.0000,  0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000,  0.0000,  0.0000]],

         [[ 0.8222, -0.0000,  0.0000,  0.0000, -0.0000],
          [ 0.1871,  1.4156,  0.0000,  0.0000, -0.0000],
          [ 0.0000, -0.

### cross-attention mask detail


In [87]:
def get_src_trg_mask(src_ids, trg_ids, pad_token_id = 0):
    bs, src_seq_len = src_ids.shape
    bs, trg_seq_len = trg_ids.shape
    
    mask = torch.ones(bs, trg_seq_len, src_seq_len) # tril
    for i in range(bs):
        src_pad_idx =  torch.where(src_ids[i, :]  == 0)[0]
        trg_pad_idx =  torch.where(trg_ids[i, :]  == 0)[0]
        mask[i, trg_pad_idx, :] = 0
        mask[i, :, src_pad_idx] = 0
    return mask
    
src_ids = torch.tensor([[1, 2, 3, 0, 0],
                          [1, 2, 0, 0, 0]], dtype = torch.long) # 0 is pad

trg_ids = torch.tensor([[4, 5, 0, 0, ],
                          [4, 5, 6, 0, ]], dtype = torch.long) # 0 is pad

mask = get_src_trg_mask(src_ids, trg_ids, 0)
print(mask)

score = torch.randn(bs, 4, 5)
score * mask

# 头并行同理

tensor([[[1., 1., 1., 0., 0.],
         [1., 1., 1., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[1., 1., 0., 0., 0.],
         [1., 1., 0., 0., 0.],
         [1., 1., 0., 0., 0.],
         [0., 0., 0., 0., 0.]]])


tensor([[[ 0.3150,  0.8006,  0.6434, -0.0000, -0.0000],
         [-1.0158,  0.4890, -0.1554,  0.0000,  0.0000],
         [-0.0000, -0.0000,  0.0000,  0.0000, -0.0000],
         [ 0.0000,  0.0000,  0.0000, -0.0000, -0.0000]],

        [[ 0.8679, -0.4921, -0.0000, -0.0000, -0.0000],
         [ 2.1011, -0.0628, -0.0000,  0.0000,  0.0000],
         [ 0.8966, -0.1124, -0.0000,  0.0000, -0.0000],
         [-0.0000, -0.0000, -0.0000,  0.0000, -0.0000]]])

## Transformer output layer

In [93]:
class TransformerOutputLayer(nn.Module):
    """
    """
    def __init__(self, vocab_size = 100, dim = 512):
        super().__init__()
        self.lm_head = nn.Linear(dim, vocab_size)
        self.softmax = nn.Softmax(dim = -1)

    def forward(self, X):
        logits = self.lm_head(X)
        prob = self.softmax(logits)
        return logits

X = torch.randn(2, 8, tmp_dim)
model = TransformerOutputLayer(trg_vocab_size, tmp_dim)
print(model)
Y = model(X)
print(X.shape, Y.shape)

TransformerOutputLayer(
  (lm_head): Linear(in_features=16, out_features=200, bias=True)
  (softmax): Softmax(dim=-1)
)
torch.Size([2, 8, 16]) torch.Size([2, 8, 200])
